# Inference: Anomaly Detection (Power Consumption)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import matplotlib.pyplot as plt

from src import download_power, load_power_npz, load_checkpoint, evaluate, detect

## 1. Load Data & Model

In [ ]:
DATA_PATH = download_power("../../assets/data/power/power_processed_data.npz")

In [ ]:
train_loader, val_loader, test_loader, meta = load_power_npz(
    DATA_PATH, batch_size=64
)

print(f"Features: {meta['n_features']}  |  Seq len: {meta['seq_len']}")
print(f"Test set: {'yes' if meta['has_test'] else 'no'}")
print(f"Labels:   {'yes' if meta.get('has_labels') else 'no (unsupervised)'}")

In [ ]:
# - Pick one

# Lightweight checkpoint
CHECKPOINT = "../../checkpoints/power/lightweight/best_model.pt"
MODEL_TYPE = "lightweight"

# Full checkpoint
# CHECKPOINT = "../../checkpoints/power/full/best_model.pt"
# MODEL_TYPE = "full"

model = load_checkpoint(
    CHECKPOINT,
    model_type=MODEL_TYPE,
    input_dim=meta["n_features"],
    preset="medium",
)

## 2. Evaluate

No anomaly labels exist for this dataset, so AUC / F1 are not available.  
We report reconstruction error statistics instead.

In [ ]:
if test_loader:
    eval_results = evaluate(model, test_loader)
    scores = eval_results["scores"]
    print(f"Mean reconstruction error: {scores.mean():.6f}")
    print(f"Std reconstruction error:  {scores.std():.6f}")
    print(f"Min: {scores.min():.6f}  |  Max: {scores.max():.6f}")
else:
    print("No test set available.")

## 3. Detect Anomalies

In [ ]:
test_data = np.concatenate([b[0].numpy() for b in test_loader], axis=0)

results = detect(
    model,
    test_data,
    threshold_percentile=95.0,
)

print(f"Threshold:    {results['threshold']:.6f}")
print(f"Anomalies:    {results['n_anomalies']} / {len(results['scores'])}")
print(f"Anomaly rate: {results['anomaly_rate'] * 100:.2f}%")

## 4. Visualize

In [ ]:
scores = results["scores"]
threshold = results["threshold"]
anomalies = results["anomalies"]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(scores, linewidth=0.8, alpha=0.8)
axes[0].axhline(threshold, color="red", ls="--", label=f"Threshold ({threshold:.4f})")
axes[0].fill_between(
    range(len(scores)), scores, threshold,
    where=anomalies, alpha=0.3, color="red",
)
axes[0].set_ylabel("Reconstruction Error")
axes[0].set_title("Per-Window Anomaly Scores (Power Consumption)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(
    np.where(~anomalies)[0], scores[~anomalies],
    c="steelblue", s=3, alpha=0.3, label="Normal",
)
axes[1].scatter(
    np.where(anomalies)[0], scores[anomalies],
    c="red", s=10, label="Anomaly",
)
axes[1].axhline(threshold, color="red", ls="--", alpha=0.5)
axes[1].set_xlabel("Window Index")
axes[1].set_ylabel("Score")
axes[1].set_title("Anomaly Classification")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores[~anomalies], bins=50, alpha=0.7, label="Normal", color="steelblue")
ax.hist(scores[anomalies], bins=20, alpha=0.7, label="Anomaly", color="red")
ax.axvline(threshold, color="red", ls="--", lw=2, label="Threshold")
ax.set_xlabel("Reconstruction Error")
ax.set_ylabel("Count")
ax.set_title("Score Distribution (Power Consumption)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()